# Round Summary Generator

Generates `prior_summary_gold` values for each game round using Claude Sonnet 4.5.

**Usage**: Set `ROUND_ID` at the top of the Configuration cell. Run all cells top-to-bottom.

| Round | Data Source | Output |
|---|---|---|
| 1 | `Deception-Dataset.csv` (filtered to R1) | `Datasets/summarizer/summaries_r1.csv` |
| 2 | `Datasets/verified/verified_r2_seeds_combined.csv` | `Datasets/summarizer/summaries_r2.csv` |
| 3 | `Datasets/verified/verified_r3_seeds_combined.csv` | `Datasets/summarizer/summaries_r3.csv` |
| 4 | `Datasets/verified/verified_r4_seeds_combined.csv` | `Datasets/summarizer/summaries_r4.csv` |
| 5 | `Datasets/verified/verified_r5_seeds_combined.csv` | `Datasets/summarizer/summaries_r5.csv` |

Each round is summarized standalone — only that round's data is used. After reviewing the output, manually merge the summaries into `Deception-Dataset.csv` under `prior_summary_gold` (R2 rows receive the R1 summary, R3 rows receive the concatenated R1+R2 summaries, etc.).

## ⚙️ Configuration — Change `ROUND_ID` Here

In [ ]:
import os

# ─── CHANGE ONLY THIS ────────────────────────────────────────────────────────
ROUND_ID   = 3        # 1, 2, 3, 4, or 5
TEST_LIMIT = 'full'   # 'full' to process all games, or an integer (e.g. 3) for a test run
# ─────────────────────────────────────────────────────────────────────────────

# R1 lives in the master dataset (verified_r1 is missing the first 25 games)
# R2+ each have their own verified file
if ROUND_ID == 1:
    DATA_SOURCE = 'Deception-Dataset.csv'
else:
    DATA_SOURCE = f'Datasets/verified/verified_r{ROUND_ID}_seeds_combined.csv'

OUTPUT_DIR  = 'Datasets/summarizer'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f'summaries_r{ROUND_ID}.csv')

print(f"Round:        {ROUND_ID}")
print(f"Data source:  {DATA_SOURCE}")
print(f"Output:       {OUTPUT_FILE}")
print(f"Test limit:   {TEST_LIMIT}")

# Validate data source exists
assert os.path.exists(DATA_SOURCE), f"Data source not found: {DATA_SOURCE}"
print(f"\n✓ Data source found")

Round:        1
Data source:  Deception-Dataset.csv
Output:       Datasets/summarizer\summaries_r1.csv
Test limit:   full

✓ Data source found


## Load Data

In [65]:
import pandas as pd
import json

# R1: filter from master dataset (verified_r1_seeds_combined only has 225/250 games)
# R2+: read the verified file directly (already round-scoped)
if ROUND_ID == 1:
    _df = pd.read_csv(DATA_SOURCE)
    round_df = _df[_df['round_id'] == 1].copy().reset_index(drop=True)
else:
    round_df = pd.read_csv(DATA_SOURCE)

round_df = round_df[
    round_df['discussion_log'].notna() &
    (round_df['discussion_log'].astype(str).str.strip() != '')
].reset_index(drop=True)

print(f"R{ROUND_ID} rows with discussion_log: {len(round_df)}")
print(f"Columns: {list(round_df.columns)}")

# Preview a sample row
row = round_df.iloc[0]
print(f"\nSample game: {row['game_id']} | Round: {row.get('round_id', ROUND_ID)}")
print(f"public_history:\n{row['public_history']}")
print(f"discussion_log (first 500 chars): {str(row['discussion_log'])[:500]}")

R1 rows with discussion_log: 250
Columns: ['game_id', 'round_id', 'role_id', 'llm_alignment', 'player_roles', 'public_history', 'prior_summary_gold', 'discussion_log', 'matrix_tactic_scale', 'reasoning_gold', 'Overall_with_formula', 'Overall']

Sample game: G001 | Round: 1
public_history:
Round: 1
Leader: P4
Team: P1, P4
Votes: P1:Y P2:Y P3:Y P4:Y P5:Y
Quest 1 Outcome: FAIL
discussion_log (first 500 chars): Discussion after Quest 1:

P3: "I think that P1, P4, and me would make a great team"

P2: "P1 and P4 went on the last mission and it failed. P3 must be evil"

P4: "I know the first mission with me and P1 failed, but it was obviously P1 who failed it"

P5: "It is not a good idea to form a team with both P1 and P4."



## Initialize Claude Sonnet 4.5 Client

In [66]:
import anthropic
from dotenv import load_dotenv
import time

load_dotenv()
client = anthropic.Anthropic(max_retries=5)

SYSTEM_PROMPT = (
    "You are an expert Avalon game analyst. You generate accurate, concise round summaries "
    "that capture key behavioral patterns and discussion dynamics from completed rounds. "
    "Your summaries are used as prior context for future round — "
    "they must convey what happened without losing critical social dynamics."
)

print("✓ Claude Sonnet 4.5 client initialized (model: claude-sonnet-4-5-20250929)")

✓ Claude Sonnet 4.5 client initialized (model: claude-sonnet-4-5-20250929)


## Summary Generation Function

**Design choices:**
- Player roles and tactic annotations are provided as input context so Claude understands the strategic dynamics, but are explicitly prohibited from appearing in the output.
- **No alignment disclosure**: summaries never reveal which players are Good or Evil — they describe behavior only. This is critical because the `prior_summary_gold` column may be used to train agents who must not learn alignments from the summary.
- Tactic label names are banned from the output; Claude must paraphrase in plain behavioral language.
- The static instructions are sent with `cache_control: ephemeral` so all subsequent calls in the same run are served from Anthropic's prompt cache (10% of input token cost).
- Target: 3–4 narrative sentences after a `Round X Summary:` header line, 80–140 words.

In [67]:
def _format_tactic_annotations(player_roles_str: str, matrix_tactic_scale_str: str) -> str:
    """Format tactic annotations with alignment context for Claude's reference."""
    try:
        roles = json.loads(player_roles_str) if isinstance(player_roles_str, str) else player_roles_str
        tactics = json.loads(matrix_tactic_scale_str) if isinstance(matrix_tactic_scale_str, str) else matrix_tactic_scale_str
        lines = []
        for player, info in tactics.items():
            alignment = roles.get(player, '?')
            tactic = info.get('tactic', '?')
            scale  = info.get('scale', '?')
            level  = info.get('level', '?')
            lines.append(f"  {player} ({alignment}): {tactic} | {scale} {level}")
        return "\n".join(lines)
    except Exception:
        return str(matrix_tactic_scale_str)


# ─── Static instructions — built once, cached across all API calls ────────────
# cache_control: ephemeral marks the end of the cached prefix. After the first
# call, all subsequent calls pay only 10% of input token cost for this block.
STATIC_SUMMARY_PROMPT = f"""You are generating a concise round summary for an Avalon social deduction game.
This summary will be stored as prior context for agents in future rounds.
It captures what happened, who led, which team was proposed, how voting went, the quest outcome, and key behavioral dynamics in a scannable, narrative format.

REQUIRED SUMMARY FORMAT:

Line 1: Round {{ROUND_ID}} Summary:
Line 2: Leader: [player] | Team: [players] | Vote: [vote result] | Outcome: [PASS/FAIL]
Line 3: [2~3 brief sentences of discussion dynamics as a paragraph — no brackets, no labels, no linebreaks, plain narrative prose]

STRUCTURAL LINE RULES (Line 2):
- Leader: Name the player who led the round (e.g., "Leader: P1").
- Team: List the proposed team members separated by commas (e.g., "Team: P1, P4") — just player IDs, no descriptions.
- Vote: Describe the vote result in concise natural language (e.g., "Unanimous", "Approved but P1 voted no").
- Outcome: PASS if the quest passed, FAIL if it failed. Use only these two words.

- DISCUSSION DYNAMICS (Line 3: 2~3 brief sentences):
- Sentence 1: Describe how the dynamics unfolded—who supported or challenged whom, what evidence or arguments were raised, and any shifts in suspicion or consensus. Be sure to capture all key developments for each player.
- Sentence 2: Describe the enduring conflict or consensus that emerged — which player relationships became tense or strengthened, what will likely influence the next round's discussions.

WRITING RULES:
- BEHAVIORAL LANGUAGE: Describe all behavior in plain natural language. Do NOT use tactic annotation label names verbatim (e.g., instead of "Deflection", write "deflected blame onto P3"; instead of "Reverse accusation", write "turned suspicion back onto P1"; instead of "Evidence sharing", write "cited the vote record to argue P2 was trustworthy").
- NO ALIGNMENT DISCLOSURE: Never reveal any player's alignment (Good or Evil) — not as labels, not by implication. Do NOT write "P4 (Evil)", "the Evil player", "P2 who is Good", "when P1, an Evil player", or any phrasing that lets a reader deduce alignments. Do NOT infer or state suspected alignments based on the tactic annotations. Keep the summary focused solely on observed behavior and stated claims.
- AFTER A FAILED QUEST: Emphasize who was blamed for the failure (usually the mission members), how they responded, and which players protested the blame. Note any shifts in trust or alliance.
- AFTER A SUCCESSFUL QUEST: Focus on whether players celebrated or remained skeptical of the team, who pushed to keep the same team versus rotate, and what logic they used.
- PRONOUNS: Use player IDs (P1, P2, etc.) specifically instead of pronouns to avoid ambiguity.
- LENGTH: Line 1 + Line 2 + Line 3 (2~3 brief sentences). Target 50–80 words total. No bullet points, no inline metadata, no extra labels.

Return ONLY the summary text for the following Round {{ROUND_ID}} information below, strictly following the format and rules above. Do not include any explanation or extra text."""


def summarize_round(
    game_id: str,
    round_id: int,
    player_roles: str,
    public_history: str,
    discussion_log: str,
    matrix_tactic_scale: str,
) -> str | None:
    """
    Generate a round summary with structural metadata and discussion dynamics.
    Returns the summary string (starting with 'Round X Summary:'), or None on API error.
    """
    formatted_tactics = _format_tactic_annotations(player_roles, matrix_tactic_scale)

    dynamic_part = f"""Game: {game_id}

Player Roles (background reference — DO NOT disclose alignments in the summary):
{player_roles}

Public History — Round {round_id}:
{public_history}

Discussion Log — Round {round_id}:
{discussion_log}

Tactic Annotations (behavioral reference ONLY — DO NOT use these label names in the summary):
{formatted_tactics}"""

    try:
        response = client.messages.create(
            model="claude-sonnet-4-5-20250929",
            max_tokens=512,
            system=SYSTEM_PROMPT,
            messages=[{
                "role": "user",
                "content": [
                    # Static instructions: format rules, writing constraints — cached after first call
                    {"type": "text", "text": STATIC_SUMMARY_PROMPT, "cache_control": {"type": "ephemeral"}},
                    # Dynamic game data: unique per call, never cached
                    {"type": "text", "text": dynamic_part},
                ]
            }]
        )
        return response.content[0].text.strip()
    except Exception as e:
        print(f"    ✗ API error for {game_id}/R{round_id}: {e}")
        return None


print(f"✓ STATIC_SUMMARY_PROMPT built ({len(STATIC_SUMMARY_PROMPT)} chars)")
print("✓ summarize_round() defined")

✓ STATIC_SUMMARY_PROMPT built (3014 chars)
✓ summarize_round() defined


## Summarization Loop

Set `TEST_LIMIT = 'full'` in the Configuration cell to process all games, or set it to an integer to process that many. Saves incrementally every 25 games.

In [68]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

results: list[dict] = []
error_indices: list[dict] = []

limit = len(round_df) if TEST_LIMIT == 'full' else min(int(TEST_LIMIT), len(round_df))

print(f"Summarizing {limit} R{ROUND_ID} games → {OUTPUT_FILE}")
print("=" * 70)

for i in range(limit):
    row     = round_df.iloc[i]
    game_id = row['game_id']

    print(f"[{i+1:>3}/{limit}] {game_id}/R{ROUND_ID}...", end=" ", flush=True)

    summary = summarize_round(
        game_id             = game_id,
        round_id            = ROUND_ID,
        player_roles        = str(row['player_roles']),
        public_history      = str(row['public_history']),
        discussion_log      = str(row['discussion_log']),
        matrix_tactic_scale = str(row['matrix_tactic_scale']),
    )

    if summary:
        results.append({'game_id': game_id, 'round_id': ROUND_ID, 'round_summary': summary})
        print("✓")
    else:
        error_indices.append({'game_id': game_id, 'idx': i})
        print("✗ queued for retry")

    # Incremental save every 25 games
    if (i + 1) % 25 == 0 and results:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
        print(f"  💾 Interim save ({len(results)} summaries) → {OUTPUT_FILE}")

    time.sleep(0.3)  # gentle rate limiting

print("=" * 70)
print(f"Done: {len(results)} succeeded, {len(error_indices)} failed")
if error_indices:
    print(f"Failed games: {[e['game_id'] for e in error_indices]}")

Summarizing 250 R1 games → Datasets/summarizer\summaries_r1.csv
[  1/250] G001/R1... ✓
[  2/250] G002/R1... ✓
[  3/250] G003/R1... ✓
[  4/250] G004/R1... ✓
[  5/250] G005/R1... ✓
[  6/250] G006/R1... ✓
[  7/250] G007/R1... ✓
[  8/250] G008/R1... ✓
[  9/250] G009/R1... ✓
[ 10/250] G010/R1... ✓
[ 11/250] G011/R1... ✓
[ 12/250] G012/R1... ✓
[ 13/250] G013/R1... ✓
[ 14/250] G014/R1... ✓
[ 15/250] G015/R1... ✓
[ 16/250] G016/R1... ✓
[ 17/250] G017/R1... ✓
[ 18/250] G018/R1... ✓
[ 19/250] G019/R1... ✓
[ 20/250] G020/R1... ✓
[ 21/250] G021/R1... ✓
[ 22/250] G022/R1... ✓
[ 23/250] G023/R1... ✓
[ 24/250] G024/R1... ✓
[ 25/250] G025/R1... ✓
  💾 Interim save (25 summaries) → Datasets/summarizer\summaries_r1.csv
[ 26/250] G026/R1... ✓
[ 27/250] G027/R1... ✓
[ 28/250] G028/R1... ✓
[ 29/250] G029/R1... ✓
[ 30/250] G030/R1... ✓
[ 31/250] G031/R1... ✓
[ 32/250] G032/R1... ✓
[ 33/250] G033/R1... ✓
[ 34/250] G034/R1... ✓
[ 35/250] G035/R1... ✓
[ 36/250] G036/R1... ✓
[ 37/250] G037/R1... ✓
[ 38/250] G038

## Retry Failed Games (if any)

In [69]:
if not error_indices:
    print("✅ No errors to retry.")
else:
    print(f"Retrying {len(error_indices)} game(s)...")
    still_failed = []

    for entry in error_indices:
        game_id = entry['game_id']
        idx     = entry['idx']
        row     = round_df.iloc[idx]

        print(f"  Retrying {game_id}...", end=" ", flush=True)
        time.sleep(2)

        summary = summarize_round(
            game_id             = game_id,
            round_id            = ROUND_ID,
            player_roles        = str(row['player_roles']),
            public_history      = str(row['public_history']),
            discussion_log      = str(row['discussion_log']),
            matrix_tactic_scale = str(row['matrix_tactic_scale']),
        )

        if summary:
            results.append({'game_id': game_id, 'round_id': ROUND_ID, 'round_summary': summary})
            print("✓")
        else:
            still_failed.append(game_id)
            print("✗ still failed")

    error_indices = [{'game_id': g, 'idx': None} for g in still_failed]
    print(f"\nRetry complete: {len(still_failed)} still failed")
    if still_failed:
        print("  Unresolved:", still_failed)
        print("  These games will be absent from the output CSV — add manually if needed.")

✅ No errors to retry.


## Save Output

In [70]:
results_df = (
    pd.DataFrame(results)
    .sort_values('game_id')
    .reset_index(drop=True)
)

results_df.to_csv(OUTPUT_FILE, index=False)
print(f"✅ Saved {len(results_df)} summaries → {OUTPUT_FILE}")
print(f"   Columns: {list(results_df.columns)}")
print()

# Preview sample summaries
print("=" * 70)
print("SAMPLE SUMMARIES (first 5):")
print("=" * 70)
for _, row in results_df.head(5).iterrows():
    print(f"\n{row['game_id']}:")
    print(row['round_summary'])
    print(f"  [{len(row['round_summary'].split())} words]")

✅ Saved 250 summaries → Datasets/summarizer\summaries_r1.csv
   Columns: ['game_id', 'round_id', 'round_summary']

SAMPLE SUMMARIES (first 5):

G001:
Round 1 Summary:
Leader: P4 | Team: P1, P4 | Vote: Unanimous | Outcome: FAIL

After the failed quest, P4 immediately blamed P1 for the failure, while P3 proposed including both P1 and P4 in the next team alongside P3. P2 argued that since P1 and P4 were both on the failed mission, P3 must be untrustworthy for wanting to team with them, and P5 warned against keeping P1 and P4 together. The round established early tension between P4 and P1 over responsibility for the failure, while P2 and P5 both voiced skepticism toward different players' team-building logic.
  [98 words]

G002:
Round 1 Summary:
Leader: P4 | Team: P1, P4 | Vote: Unanimous approval | Outcome: FAIL

P3 immediately pointed to the mission members as suspect and urged excluding both P1 and P4 from future teams, while P2 suggested building a fresh team with P2, P3, and P5. P5 ac

## QA Review

Check word count distribution and flag summaries outside target range (80–140 words).

In [73]:
import re

results_df['word_count']     = results_df['round_summary'].apply(lambda x: len(x.split()))
results_df['sentence_count'] = results_df['round_summary'].apply(
    lambda x: len([s for s in re.split(r'[.!?]+', x.strip()) if s.strip()])
)

print("Word count stats:")
print(results_df['word_count'].describe().round(1).to_string())

print("\nSentence count distribution:")
print(results_df['sentence_count'].value_counts().sort_index().to_string())

# Flag outliers
too_short = results_df[results_df['word_count'] < 40]
too_long  = results_df[results_df['word_count'] > 140]

if len(too_short) > 0:
    print(f"\n⚠️  {len(too_short)} summaries are too short (<40 words):")
    for _, r in too_short.iterrows():
        print(f"  {r['game_id']}: {r['word_count']} words")

if len(too_long) > 0:
    print(f"\n⚠️  {len(too_long)} summaries are too long (>140 words):")
    for _, r in too_long.iterrows():
        print(f"  {r['game_id']}: {r['word_count']} words")

if len(too_short) == 0 and len(too_long) == 0:
    print("\n✅ All summaries within target range (40–140 words).")

print(f"\n✅ Summaries ready for review at: {OUTPUT_FILE}")
print()
print("Next steps:")
print(f"  1. Review sample summaries above for quality")
print(f"  2. Set ROUND_ID = {ROUND_ID + 1} and re-run this notebook for the next round")
print(f"  3. After all rounds done, manually merge into Deception-Dataset.csv:")
print(f"     - R{ROUND_ID + 1} rows: prior_summary_gold = R{ROUND_ID} round_summary")

Word count stats:
count    250.0
mean     102.9
std       10.7
min       76.0
25%       95.2
50%      102.0
75%      109.8
max      128.0

Sentence count distribution:
sentence_count
2      4
3    238
4      8

✅ All summaries within target range (40–140 words).

✅ Summaries ready for review at: Datasets/summarizer\summaries_r1.csv

Next steps:
  1. Review sample summaries above for quality
  2. Set ROUND_ID = 2 and re-run this notebook for the next round
  3. After all rounds done, manually merge into Deception-Dataset.csv:
     - R2 rows: prior_summary_gold = R1 round_summary
